In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import mode


In [ ]:
df_movies = pd.read_parquet("processed_data/movies_data_updated.parquet")
df_ratings = pd.read_parquet("processed_data/ratings_data_extended.parquet")

In [ ]:
df_movies["director"].isna().sum()

In [ ]:
df_movies = df_movies.drop(columns=['overview', 'tagline', 'runtime_bin', 'homepage',
       'collection_name', 'roi', 'log_profit', 'tag', 'producer',
       'producer_popularity', 'other_lead', 'other_lead_popularity',
       'other_lead_genre_diversity', 'main_language', 'main_country',
       'main_production_company', 'company_popularity', 'franchise_strength','other_actors', 'spoken_language_names',
       'has_translation', 'production_country_names',
       'production_company_names'])
df_movies.shape

In [ ]:
df_movies.shape

In [ ]:
df_movies = df_movies.loc[df_movies["director"].isna() == False]
print(f"After removing nan director: {df_movies.shape}")
df_movies = df_movies.loc[df_movies["release_date"].isna() == False]
print(f"After removing nan release_date: {df_movies.shape}")
df_movies = df_movies.loc[df_movies["lead_actor"].isna() == False]
print(f"After removing nan lead_actor: {df_movies.shape}")
df_movies = df_movies.loc[df_movies["runtime"].isna() == False]
print(f"After removing nan runtime: {df_movies.shape}")
df_movies = df_movies.loc[df_movies["status"] == "Released"]
print(f"After keeping only released movies (status column): {df_movies.shape}")
df_movies = df_movies.loc[df_movies["vote_average"].isna() == False]
print(f"After removing nan vote_average: {df_movies.shape}")
df_movies = df_movies.loc[df_movies["original_language"].isna() == False]
print(f"After removing nan original_language: {df_movies.shape}")
df_movies = df_movies.loc[df_movies["poster_url"].isna() == False]
print(f"After removing nan poster_url: {df_movies.shape}")
df_movies = df_movies.loc[df_movies["adult"] == "False"]
print(f"After removing adult movies: {df_movies.shape}")

df_movies["main_genre"].replace({"Sci-Fi": "Science Fiction", "Musical": "Music", "Children": "Fantasy", "History": "Documentary"}, inplace=True)
df_movies = df_movies.loc[~(df_movies["main_genre"].isin(["Film-Noir", "Foreign"]))]
print(f"After Fixing and removing some main_genre: {df_movies.shape}")
df_movies = df_movies.loc[~(df_movies["main_genre"].str.contains("no genres listed"))]
print(f"After removing no main_genre: {df_movies.shape}")

dfx = pd.DataFrame(df_movies["original_language"].value_counts()).reset_index(drop=False)
skip_low_original_language_movies = dfx.loc[dfx["count"] < 500]["original_language"].values
df_movies = df_movies.loc[~(df_movies["original_language"].isin(skip_low_original_language_movies))]
print(f"After removing language movies with very few counts: {df_movies.shape}")



df_movies = df_movies.drop_duplicates(subset=["movieId"])
print(f"After removing duplicate movieId: {df_movies.shape}")


In [ ]:
df_movies["main_genre"].value_counts()

In [ ]:
df_movies = df_movies.drop(columns=['budget', 'revenue','status', 'adult','profit', 'log_budget', 'log_revenue', 'imdbId', 'tmdbId', 'imdb_id', "is_recent",
                                    'vote_average', 'vote_count',
       'vote_min', 'vote_max','popularity_score',
       'critical_success', 'crowd_approval','director_popularity','lead_actor_popularity',
       'lead_actor_genre_diversity', 'is_short_film', 'is_feature_film', 'is_long_film'
       ])

# Recompute columns after removing movies


In [ ]:
considered_movies = list(df_movies["movieId"].unique())
print(df_ratings.shape)
df_ratings = df_ratings.loc[df_ratings["movieId"].isin(considered_movies)]
print(df_ratings.shape)

In [ ]:
rating_stats = df_ratings.groupby('movieId')['rating'].agg(['mean', 'min', 'max', 'count']).reset_index()
rating_stats.columns = ['movieId', 'vote_average', 'vote_min', 'vote_max', 'vote_count']

In [ ]:
len(rating_stats["movieId"].unique())

## Select movies that have ratings only


In [ ]:
movies_with_ratings = rating_stats["movieId"].unique()
df_movies = df_movies.loc[df_movies["movieId"].isin(movies_with_ratings)]
df_movies.shape

## Calculating values


In [ ]:
df_movies = pd.merge(df_movies, rating_stats, on='movieId', how='left')

In [ ]:
df_movies['popularity_score'] = df_movies['vote_average'] * np.log1p(df_movies['vote_count'])
df_movies["critical_success"] = df_movies[["vote_average", "popularity_score"]].mean(axis=1)
df_movies["crowd_approval"] = df_movies["vote_average"] * np.log1p(df_movies["vote_count"])

In [ ]:
actor_popularity = (
    df_movies.groupby("lead_actor")
    .agg({
        "critical_success": "mean",
        "crowd_approval": "mean",
        "vote_average": "mean",
        "vote_count": "sum",
        "popularity_score": "sum",
    })
    .fillna(0)
)

scaler = MinMaxScaler()
actor_popularity_scaled = pd.DataFrame(
    scaler.fit_transform(actor_popularity),
    columns=actor_popularity.columns,
    index=actor_popularity.index
)

actor_popularity_scaled["popularity_index"] = (
    0.4 * actor_popularity_scaled["critical_success"] +
    0.3 * actor_popularity_scaled["crowd_approval"] +
    0.1 * actor_popularity_scaled["vote_average"] +
    0.1 * actor_popularity_scaled["vote_count"] +
    0.1 * actor_popularity_scaled["popularity_score"]
)

df_movies["lead_actor_popularity"] = df_movies["lead_actor"].map(actor_popularity_scaled["popularity_index"])


In [ ]:
director_popularity = (
    df_movies.groupby("director")
    .agg({
        "critical_success": "mean",
        "crowd_approval": "mean",
        "vote_average": "mean",
        "vote_count": "sum",
        "popularity_score": "sum",
    })
    .fillna(0)
)

scaler = MinMaxScaler()
director_popularity_scaled = pd.DataFrame(
    scaler.fit_transform(director_popularity),
    columns=director_popularity.columns,
    index=director_popularity.index
)

director_popularity_scaled["popularity_index"] = (
    0.4 * director_popularity_scaled["critical_success"] +
    0.3 * director_popularity_scaled["crowd_approval"] +
    0.1 * director_popularity_scaled["vote_average"] +
    0.1 * director_popularity_scaled["vote_count"] +
    0.1 * director_popularity_scaled["popularity_score"]
)

df_movies["director_popularity"] = df_movies["director"].map(director_popularity_scaled["popularity_index"])


# DF Ratings setup


In [ ]:
df_ratings.head()

In [ ]:
print(df_ratings.shape)
df_ratings.drop_duplicates(subset=["movieId", "userId"], keep = "first", inplace = True)
print(df_ratings.shape)

In [ ]:
user_stats = df_ratings.groupby('userId').agg(
    num_ratings=('movieId', 'count'),
    mean_rating=('rating', 'mean'),
    std_rating=('rating', 'std'),
    min_rating=('rating', 'min'),
    max_rating=('rating', 'max'),
).reset_index()

def rating_counts(group):
    min_r = group['rating'].min()
    max_r = group['rating'].max()
    mode_r = group['rating'].mode()[0]

    return pd.Series({
        'min_rating_count': (group['rating'] == min_r).sum(),
        'max_rating_count': (group['rating'] == max_r).sum(),
        'mode_rating': mode_r,
        'mode_rating_count': (group['rating'] == mode_r).sum()
    })

rating_distribution = df_ratings.groupby('userId').apply(rating_counts).reset_index()

user_stats = pd.merge(user_stats, rating_distribution, on='userId')

In [ ]:
genre_counts = df_ratings.groupby(['userId', 'main_genre']).size().reset_index(name='count')

genre_counts['rank'] = genre_counts.groupby('userId')['count'].rank(method='first', ascending=False)

num_genre = 3

top_n_genres = genre_counts[genre_counts['rank'] <= num_genre].copy()

top_num_genre_lists = (
    top_n_genres.sort_values(['userId', 'rank'])
    .groupby('userId')['main_genre']
    .apply(lambda x: list(x))
    .reset_index(name='top_genres')
)

for i in range(num_genre):
    top_num_genre_lists[f'top_genre_{i+1}'] = top_num_genre_lists['top_genres'].apply(lambda x: x[i] if i < len(x) else None)

user_stats = user_stats.merge(top_num_genre_lists.drop(columns='top_genres'), on='userId', how='left')
user_stats['genre_diversity'] = df_ratings.groupby('userId')['main_genre'].nunique().reset_index(drop=True)

top_genres_long = pd.melt(
    user_stats,
    id_vars=['userId'],
    value_vars=[f'top_genre_{i}' for i in range(1, num_genre+1)],
    var_name='top_genre_rank',
    value_name='genre'
).dropna()

df_user_genre = pd.merge(df_ratings, top_genres_long, left_on=['userId', 'main_genre'], right_on=['userId', 'genre'])

genre_rating_stats = df_user_genre.groupby(['userId', 'top_genre_rank']).agg(
    genre_rating_mean=('rating', 'mean'),
    genre_rating_count=('rating', 'count'),
    genre_rating_min=('rating', 'min'),
    genre_rating_max=('rating', 'max'),
).reset_index()

genre_rating_mean_wide = genre_rating_stats.pivot(index='userId', columns='top_genre_rank', values='genre_rating_mean')
genre_rating_count_wide = genre_rating_stats.pivot(index='userId', columns='top_genre_rank', values='genre_rating_count')
genre_rating_min_wide = genre_rating_stats.pivot(index='userId', columns='top_genre_rank', values='genre_rating_min')
genre_rating_max_wide = genre_rating_stats.pivot(index='userId', columns='top_genre_rank', values='genre_rating_max')

genre_rating_mean_wide.columns = [f'{col}_mean' for col in genre_rating_mean_wide.columns]
genre_rating_count_wide.columns = [f'{col}_count' for col in genre_rating_count_wide.columns]
genre_rating_min_wide.columns = [f'{col}_min' for col in genre_rating_min_wide.columns]
genre_rating_max_wide.columns = [f'{col}_max' for col in genre_rating_max_wide.columns]

user_stats = user_stats.merge(genre_rating_mean_wide, on='userId', how='left')
user_stats = user_stats.merge(genre_rating_count_wide, on='userId', how='left')
user_stats = user_stats.merge(genre_rating_min_wide, on='userId', how='left')
user_stats = user_stats.merge(genre_rating_max_wide, on='userId', how='left')



In [ ]:
# release_stats = df_ratings.groupby('userId')['release_year'].agg(['mean', 'min', 'max']).reset_index()
# release_stats['year_span'] = release_stats['max'] - release_stats['min']
# release_stats = release_stats.rename(columns={'mean': 'avg_release_year'})
# user_stats = user_stats.merge(release_stats[['userId', 'avg_release_year', 'year_span']], on='userId', how='left')

# Filtering by num ratings and saving


In [ ]:
min_ratings_count = 200

In [ ]:
user_stats = user_stats.loc[user_stats["num_ratings"] > min_ratings_count]

In [ ]:
df_ratings = df_ratings.loc[df_ratings["userId"].isin(user_stats["userId"].unique())]
df_ratings = df_ratings.loc[df_ratings["userId"].isin(user_stats.loc[user_stats["num_ratings"] > min_ratings_count]["userId"].unique())]

In [ ]:
df_movies = df_movies.loc[df_movies["movieId"].isin(df_ratings["movieId"].unique())]
df_movies.to_parquet("processed_data/movies_filtered_cleaned.parquet", index = False)
df_ratings.to_parquet("processed_data/processed_df_ratings_filtered.parquet", index = False)
user_stats.to_parquet("processed_data/all_users_stats_post_movies_filter.parquet", index = False)